# Inspect train / eval JSONL manifests
Check field coverage, image counts, candidates/target shape for FusionGVJEPA manifests.
Train rows: `{"images": [...], "query": "...", "target": "..."}`
Eval (SPAR select) rows: `{"images": [...], "query": "...", "candidates": [...], "target": "..."}`

In [ ]:
import json
from collections import Counter
from pathlib import Path
from typing import Any

import pandas as pd

In [ ]:
# --- edit these paths ---
TRAIN_PATH = Path("../data/spar_sftqa_train.jsonl")
EVAL_PATH = Path("../data/spar_sftqa_eval.jsonl")
INPUT_ROOT = None  # set if image paths in jsonl are relative, e.g. Path("..")
MAX_ROWS = None    # cap rows scanned, for huge files

In [ ]:
def type_name(value: Any) -> str:
    if isinstance(value, list):
        if value and all(isinstance(v, str) for v in value):
            return f"list[str]({len(value)})"
        return f"list({len(value)})"
    return type(value).__name__


def image_list(row: dict) -> list:
    if "images" in row:
        v = row["images"]
        return v if isinstance(v, list) else [v]
    if "image" in row:
        v = row["image"]
        return v if isinstance(v, list) else [v]
    return []


def candidate_list(row: dict) -> list:
    for key in ("candidates", "choices", "options", "multichoices"):
        if key in row:
            v = row[key]
            if isinstance(v, dict):
                v = list(v.values())
            return v if isinstance(v, list) else [v]
    return []


def load_rows(path: Path, max_rows: int | None = None) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            if max_rows is not None and len(rows) >= max_rows:
                break
            rows.append(json.loads(line))
    return rows


def inspect(rows: list[dict], input_root: Path | None = None) -> dict:
    key_counts: Counter = Counter()
    key_types: dict[str, Counter] = {}
    n_images_counts: Counter = Counter()
    candidate_len_counts: Counter = Counter()
    has_candidates = 0
    has_target = 0
    missing_images = 0
    checked_images = 0

    for row in rows:
        for k, v in row.items():
            key_counts[k] += 1
            key_types.setdefault(k, Counter())[type_name(v)] += 1

        imgs = image_list(row)
        n_images_counts[len(imgs)] += 1

        cands = candidate_list(row)
        if cands:
            has_candidates += 1
            candidate_len_counts[len(cands)] += 1
        if any(k in row for k in ("target", "answer", "label")):
            has_target += 1

        if input_root is not None:
            for img in imgs:
                p = Path(img)
                if not p.is_absolute():
                    p = input_root / p
                checked_images += 1
                if not p.exists():
                    missing_images += 1

    return {
        "total": len(rows),
        "key_counts": key_counts,
        "key_types": key_types,
        "n_images_counts": n_images_counts,
        "has_candidates": has_candidates,
        "candidate_len_counts": candidate_len_counts,
        "has_target": has_target,
        "checked_images": checked_images,
        "missing_images": missing_images,
    }


def report(name: str, path: Path, stats: dict) -> None:
    total = stats["total"]
    print(f"=== {name}: {path} ===")
    print(f"rows: {total}")
    print("key coverage:")
    for k, c in stats["key_counts"].most_common():
        types = ", ".join(f"{t}={n}" for t, n in stats["key_types"][k].most_common())
        print(f"  {k}: {c}/{total}  [{types}]")
    print(f"images-per-row: {dict(sorted(stats['n_images_counts'].items()))}")
    print(f"rows with candidates: {stats['has_candidates']}/{total}")
    if stats["candidate_len_counts"]:
        print(f"candidates-per-row: {dict(sorted(stats['candidate_len_counts'].items()))}")
    print(f"rows with target/answer/label: {stats['has_target']}/{total}")
    if stats["checked_images"]:
        ok = stats["checked_images"] - stats["missing_images"]
        print(f"image check: {ok}/{stats['checked_images']} exist, {stats['missing_images']} missing")
    print()

In [ ]:
train_rows = load_rows(TRAIN_PATH, MAX_ROWS)
train_stats = inspect(train_rows, INPUT_ROOT)
report("train", TRAIN_PATH, train_stats)

In [ ]:
eval_rows = load_rows(EVAL_PATH, MAX_ROWS)
eval_stats = inspect(eval_rows, INPUT_ROOT)
report("eval", EVAL_PATH, eval_stats)

In [ ]:
# key sets diff: fields present in one manifest but not the other
train_keys = set(train_stats["key_counts"])
eval_keys = set(eval_stats["key_counts"])
print("only in train:", train_keys - eval_keys)
print("only in eval: ", eval_keys - train_keys)
print("shared:       ", train_keys & eval_keys)

In [ ]:
# sample rows, side by side
print("train sample:")
print(json.dumps(train_rows[0], ensure_ascii=False, indent=2)[:800] if train_rows else "(empty)")
print("\neval sample:")
print(json.dumps(eval_rows[0], ensure_ascii=False, indent=2)[:800] if eval_rows else "(empty)")

In [ ]:
# quick view as dataframe (flattened, images/candidates truncated for display)
def to_frame(rows: list[dict], n: int = 20) -> pd.DataFrame:
    flat = []
    for row in rows[:n]:
        r = dict(row)
        r["num_images"] = len(image_list(row))
        r["num_candidates"] = len(candidate_list(row))
        r.pop("images", None)
        r.pop("image", None)
        flat.append(r)
    return pd.DataFrame(flat)

to_frame(eval_rows)